In [0]:
%pip install --upgrade --quiet  langchain databricks-vectorsearch tiktoken datasets

In [0]:
dbutils.library.restartPython()

In [0]:
import datasets

knowledge_base = datasets.load_dataset("niobures/wikipedia", split="train")

In [0]:
knowledge_base[0]

In [0]:
from langchain.docstore.document import Document
source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"].split("/")[1]}) for doc in knowledge_base
]

In [0]:
docs = []
for index,doc in enumerate(knowledge_base):
    docs.append({'id':index, 'text':doc["text"]})

In [0]:
# Create a catalog and schema if they do not exist
catalog_name = "data"
schema_name = "vector_store"
table_name = "niobures_wikipedia"

In [0]:
# Convert the docs variable to a Spark DataFrame
docs_df = spark.createDataFrame(docs)

# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# Write the DataFrame to a table in Unity Catalog with change data feed enabled
docs_df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

display(docs_df)

In [0]:
from databricks.vector_search.client import VectorSearchClient

vector_search_endpoint_name = f"vector_search_demo_endpoint"


vsc = VectorSearchClient()

In [0]:
vsc.create_endpoint(name=vector_search_endpoint_name, endpoint_type="STANDARD")

In [0]:
index = vsc.create_delta_sync_index(
  endpoint_name=vector_search_endpoint_name,
  source_table_name=f"{catalog_name}.{schema_name}.{table_name}",
  index_name=f"{catalog_name}.{schema_name}.{table_name}_index",
  pipeline_type="TRIGGERED",
  primary_key="id",
  embedding_source_column="text",
  embedding_model_endpoint_name="databricks-bge-large-en"
)

In [0]:
index = vsc.get_index(index_name=f"{catalog_name}.{schema_name}.{table_name}_index")
# Delta Sync Index with embeddings computed by Databricks
results = index.similarity_search(
    query_text="Eric Hooglund",
    columns=["id", "text"],
    num_results=2
    )

In [0]:
from langchain.schema import Document
from typing import List

def convert_vector_search_to_documents(results) -> List[Document]:
  column_names = []
  for column in results["manifest"]["columns"]:
      column_names.append(column)

  langchain_docs = []
  for item in results["result"]["data_array"]:
      metadata = {}
      score = item[-1]
      # print(score)
      i = 1
      for field in item[1:-1]:
          # print(field + "--")
          metadata[column_names[i]["name"]] = field
          i = i + 1
      doc = Document(page_content=str(item[0]), metadata=metadata)  # , 9)
      langchain_docs.append(doc)
  return langchain_docs

langchain_docs = convert_vector_search_to_documents(results)
langchain_docs

In [0]:
# Delta Sync Index using hybrid search, with embeddings computed by Databricks
results3 = index.similarity_search(
    query_text="Eric Hooglund",
    columns=["id", "text"],
    num_results=1,
    query_type="hybrid"
    )

results3